In [ ]:
import f1_env
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm

In [ ]:
class mlp_2hidden(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()

        self.ff1 = nn.Linear(input_dim, hidden_dim)
        self.ff2 = nn.Linear(hidden_dim, hidden_dim)
        self.ff3 = nn.Linear(hidden_dim, output_dim)

        nn.init.uniform_(self.ff1.weight, -0.001, 0.001)
        nn.init.uniform_(self.ff2.weight, -0.001, 0.001)
        nn.init.uniform_(self.ff3.weight, -0.001, 0.001)

        nn.init.uniform_(self.ff1.bias, -0.001, 0.001)
        nn.init.uniform_(self.ff2.bias, -0.001, 0.001)
        nn.init.uniform_(self.ff3.bias, -0.001, 0.001)

        self.ReLU = nn.ReLU()
    
    def forward(self, x):
        y = self.ReLU(self.ff1(x))
        y = self.ReLU(self.ff2(y))
        return self.ff3(y)

In [ ]:
GAMMA = 0.99
BUFFER_SIZE = 1000
BATCH_SIZE = 64
WARM_UP = 100
TARGET_UPDATE = 100
EXPLOR = 0.05
EPISODES = 1000

In [ ]:
class DQNAgent:
    def __init__(self, lr=1e-3, input_length=15): # TBD
        self.q_network = mlp_2hidden(input_length, 128, 4)
        self.target_network = mlp_2hidden(input_length, 128, 4)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()

        self.optimizer = torch.optim.Adam(self.q_network.parameters(), lr=lr)

        self.states = np.zeros((BUFFER_SIZE, input_length), dtype=np.float32)
        self.actions = np.zeros(BUFFER_SIZE, dtype=np.int64)
        self.rewards = np.zeros(BUFFER_SIZE, dtype=np.float32)
        self.next_states = np.zeros((BUFFER_SIZE, input_length), dtype=np.float32)
        self.dones = np.zeros(BUFFER_SIZE, dtype=np.float32)
        
        self.buffer_idx = 0  # Current position in circular buffer
        self.buffer_size = 0  # Number of valid transitions stored

    def select_action(self, state, action_mask):
        state_tensor = torch.FloatTensor(state)
        action_mask = torch.FloatTensor(action_mask)

        if np.random.rand() <= EXPLOR:
            legal = [i for i, legal in enumerate(action_mask) if legal == 1]
            return np.random.choice(legal)
        else:
            with torch.no_grad():
                q_values = self.q_network(state_tensor) + (action_mask - 1) * 1e9
                return q_values.argmax().item()
    
    def store_transition(self, state, action, reward, next_state, done):
        # Store at current index position
        self.states[self.buffer_idx] = state
        self.actions[self.buffer_idx] = action
        self.rewards[self.buffer_idx] = reward
        self.next_states[self.buffer_idx] = next_state
        self.dones[self.buffer_idx] = done
        
        # Move to next position (circular)
        self.buffer_idx = (self.buffer_idx + 1) % BUFFER_SIZE
        
        # Track actual number of transitions
        if self.buffer_size < BUFFER_SIZE:
            self.buffer_size += 1

    def update(self):
        if self.buffer_size < BATCH_SIZE:
            return

        indices = np.random.choice(self.buffer_size, BATCH_SIZE, replace=False)
        
        # Convert to tensors (now using numpy arrays instead of lists)
        states_t = torch.FloatTensor(self.states[indices])
        actions_t = torch.tensor(self.actions[indices], dtype=torch.long)
        rewards_t = torch.tensor(self.rewards[indices], dtype=torch.float32)
        next_states_t = torch.FloatTensor(self.next_states[indices])
        dones_t = torch.tensor(self.dones[indices], dtype=torch.float32)
        
        # max_a Q(S_{t+1}, a; w)
        q_taken = self.q_network(states_t)\
            .gather(1, actions_t.unsqueeze(1))\
                .squeeze()

        with torch.no_grad():
            # max_a Q(S_{t+1}, a; w-)
            q_next = self.target_network(next_states_t).max(1)[0]

            # R_{t+1} + gamma max_a Q(S_{t+1}, a; w-)
            q_target = rewards_t + GAMMA * q_next * (1 - dones_t)

        loss = ((q_target - q_taken) ** 2).mean()

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    def update_target(self):
        self.target_network.load_state_dict(self.q_network.state_dict())


In [ ]:
#years = [2023, 2024]
#gp_list = ['Australian Grand Prix', 'Monaco Grand Prix', 'Monza', 'Saudi Arabian Grand Prix']
years = [2023]
gp_list = ['Canadian Grand Prix']

def train_dqn(T=10.0, years=years, gp_list=gp_list):
    agent = DQNAgent(input_length=7)
    agent.q_network.train()

    all_returns = {gp: {year: [] for year in years} for gp in gp_list}
    action_usage = np.zeros(4)

    for year in years:
        for gp in gp_list:
            print(f'Training on {year} {gp}...')
            env = f1_env.F1Env(year, gp, agent_type='dqn')
            steps = 0
            for _ in tqdm(range(EPISODES)):
                state, _ = env.reset()
                t = 0
                done = False


                while not done:
                    action_mask = state['action_mask']

                    action = agent.select_action(state['state'], action_mask)
                    action_usage[action] += 1

                    next_state, reward, done, _ = env.step(action)

                    agent.store_transition(
                        state['state'],
                        action,
                        reward,
                        next_state['state'],
                        done
                    )

                    state = next_state
                    t += 1
                    steps += 1

                    if steps == WARM_UP:
                        agent.q_network.train()
                    if steps >= WARM_UP:
                        agent.update()
                        if steps % TARGET_UPDATE == 0:
                            agent.update_target()


                all_returns[gp][year].append(env.total_race_time.total_seconds())
                T = max(0.5, 0.99 * T)
    

    return all_returns, action_usage, agent

In [ ]:

all_returns, action_usage, agent = train_dqn()
print('done')